# Gold/USD Trading Strategy Analysis

This notebook demonstrates how to analyze Gold/USD data, backtest the momentum strategy, and visualize results.

In [ ]:
# Import libraries
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

from src.utils.config_loader import load_config
from src.data.data_fetcher import DataFetcher
from src.strategies.gold_momentum_strategy import GoldMomentumStrategy
from src.backtesting.backtest_engine import BacktestEngine

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Libraries imported successfully!")

## 1. Load Configuration and Data

In [ ]:
# Load configuration
config = load_config()
print("Configuration loaded successfully!")
print(f"Symbol: {config['trading']['symbol']}")
print(f"Initial Capital: ${config['trading']['initial_capital']:,}")

In [ ]:
# Fetch historical data
data_fetcher = DataFetcher(config)
data = data_fetcher.fetch_historical_data(
    start_date='2020-01-01',
    end_date='2024-12-31',
    interval='1d'
)

print(f"Data shape: {data.shape}")
print(f"Date range: {data.index[0]} to {data.index[-1]}")
data.head()

## 2. Exploratory Data Analysis

In [ ]:
# Basic statistics
print("Gold/USD Price Statistics:")
print(data['close'].describe())

# Plot price history
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(data.index, data['close'], linewidth=1.5)
ax.set_title('Gold/USD Historical Prices', fontsize=16)
ax.set_xlabel('Date')
ax.set_ylabel('Price (USD)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate daily returns
data['returns'] = data['close'].pct_change()

# Plot returns distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Returns over time
axes[0].plot(data.index, data['returns'], alpha=0.5)
axes[0].set_title('Daily Returns', fontsize=14)
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Returns')
axes[0].axhline(y=0, color='r', linestyle='--', alpha=0.3)
axes[0].grid(True, alpha=0.3)

# Returns distribution
axes[1].hist(data['returns'].dropna(), bins=50, alpha=0.7, edgecolor='black')
axes[1].set_title('Returns Distribution', fontsize=14)
axes[1].set_xlabel('Returns')
axes[1].set_ylabel('Frequency')
axes[1].axvline(x=0, color='r', linestyle='--', alpha=0.5)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Average Daily Return: {data['returns'].mean()*100:.3f}%")
print(f"Daily Volatility: {data['returns'].std()*100:.3f}%")
print(f"Annualized Volatility: {data['returns'].std()*np.sqrt(252)*100:.2f}%")

## 3. Strategy Indicators

In [ ]:
# Initialize strategy and calculate indicators
strategy = GoldMomentumStrategy(config)
data_with_indicators = strategy.calculate_indicators(data)

print("Indicators calculated:")
print(data_with_indicators.columns.tolist())
data_with_indicators.tail()

In [ ]:
# Plot price with moving averages
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(data_with_indicators.index, data_with_indicators['close'], 
        label='Price', linewidth=1.5, alpha=0.8)
ax.plot(data_with_indicators.index, data_with_indicators['ma_fast'], 
        label=f'Fast MA ({config["strategy"]["fast_ma"]})', linewidth=1.2, alpha=0.7)
ax.plot(data_with_indicators.index, data_with_indicators['ma_slow'], 
        label=f'Slow MA ({config["strategy"]["slow_ma"]})', linewidth=1.2, alpha=0.7)

ax.set_title('Gold/USD Price with Moving Averages', fontsize=16)
ax.set_xlabel('Date')
ax.set_ylabel('Price (USD)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot RSI
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Price
axes[0].plot(data_with_indicators.index, data_with_indicators['close'])
axes[0].set_title('Gold/USD Price', fontsize=14)
axes[0].set_ylabel('Price (USD)')
axes[0].grid(True, alpha=0.3)

# RSI
axes[1].plot(data_with_indicators.index, data_with_indicators['rsi'], color='purple')
axes[1].axhline(y=70, color='r', linestyle='--', alpha=0.5, label='Overbought')
axes[1].axhline(y=30, color='g', linestyle='--', alpha=0.5, label='Oversold')
axes[1].axhline(y=50, color='gray', linestyle=':', alpha=0.3)
axes[1].set_title('Relative Strength Index (RSI)', fontsize=14)
axes[1].set_xlabel('Date')
axes[1].set_ylabel('RSI')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Run Backtest

In [ ]:
# Run backtest
backtest = BacktestEngine(strategy, config)
results = backtest.run(data)

print("Backtest completed!")
print(f"Total trades: {len(results['trades'])}")
print(f"Final capital: ${results['final_capital']:,.2f}")

In [ ]:
# Display performance metrics
metrics = results['metrics']

print("\n" + "="*60)
print("BACKTEST PERFORMANCE METRICS")
print("="*60)

print("\n--- Basic Metrics ---")
print(f"Total Trades: {metrics.get('total_trades', 0)}")
print(f"Initial Capital: ${metrics.get('initial_capital', 0):,.2f}")
print(f"Final Capital: ${metrics.get('final_capital', 0):,.2f}")
print(f"Net Profit: ${metrics.get('net_profit', 0):,.2f}")
print(f"Total Return: {metrics.get('total_return_pct', 0):.2f}%")

print("\n--- Risk Metrics ---")
print(f"Max Drawdown: {metrics.get('max_drawdown_pct', 0):.2f}%")
print(f"Sharpe Ratio: {metrics.get('sharpe_ratio', 0):.2f}")
print(f"Sortino Ratio: {metrics.get('sortino_ratio', 0):.2f}")

print("\n--- Win/Loss Metrics ---")
print(f"Win Rate: {metrics.get('win_rate_pct', 0):.2f}%")
print(f"Profit Factor: {metrics.get('profit_factor', 0):.2f}")
print(f"Avg Win: ${metrics.get('avg_win', 0):,.2f}")
print(f"Avg Loss: ${metrics.get('avg_loss', 0):,.2f}")

## 5. Visualize Results

In [ ]:
# Plot equity curve
equity_df = results['equity_curve']

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Equity curve
axes[0].plot(equity_df['timestamp'], equity_df['equity'], linewidth=2)
axes[0].axhline(y=config['trading']['initial_capital'], 
               color='gray', linestyle='--', alpha=0.5, label='Initial Capital')
axes[0].set_title('Equity Curve', fontsize=16)
axes[0].set_ylabel('Equity ($)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Price with signals
signals_df = results['signals']
buy_signals = signals_df[signals_df['signal'] == 1]
sell_signals = signals_df[signals_df['signal'] == -1]

axes[1].plot(equity_df['timestamp'], equity_df['price'], 
            label='Gold Price', linewidth=1.5, alpha=0.8)
axes[1].scatter(buy_signals['timestamp'], buy_signals['price'], 
               marker='^', color='green', s=100, label='Buy', alpha=0.7, zorder=5)
axes[1].scatter(sell_signals['timestamp'], sell_signals['price'], 
               marker='v', color='red', s=100, label='Sell', alpha=0.7, zorder=5)
axes[1].set_title('Gold Price with Trading Signals', fontsize=16)
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Price ($)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze trades
trades_df = pd.DataFrame(results['trades'])

if len(trades_df) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # P&L distribution
    axes[0, 0].hist(trades_df['pnl'], bins=30, alpha=0.7, edgecolor='black')
    axes[0, 0].axvline(x=0, color='r', linestyle='--', alpha=0.5)
    axes[0, 0].set_title('P&L Distribution', fontsize=14)
    axes[0, 0].set_xlabel('P&L ($)')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Cumulative P&L
    cumulative_pnl = trades_df['pnl'].cumsum()
    axes[0, 1].plot(cumulative_pnl, linewidth=2)
    axes[0, 1].set_title('Cumulative P&L', fontsize=14)
    axes[0, 1].set_xlabel('Trade Number')
    axes[0, 1].set_ylabel('Cumulative P&L ($)')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Win/Loss by type
    win_loss = ['Win' if pnl > 0 else 'Loss' for pnl in trades_df['pnl']]
    pd.Series(win_loss).value_counts().plot(kind='bar', ax=axes[1, 0], alpha=0.7)
    axes[1, 0].set_title('Win vs Loss Trades', fontsize=14)
    axes[1, 0].set_xlabel('Outcome')
    axes[1, 0].set_ylabel('Count')
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    
    # Trade duration
    trades_df['duration_hours'] = trades_df['duration'].dt.total_seconds() / 3600
    axes[1, 1].hist(trades_df['duration_hours'], bins=30, alpha=0.7, edgecolor='black')
    axes[1, 1].set_title('Trade Duration Distribution', fontsize=14)
    axes[1, 1].set_xlabel('Duration (hours)')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Display trade summary
    print("\nTrade Summary:")
    print(trades_df[['entry_price', 'exit_price', 'pnl', 'pnl_pct']].describe())
else:
    print("No trades to analyze")

## 6. Strategy Parameter Optimization

Test different parameter combinations to find optimal settings.

In [ ]:
# Parameter grid for optimization
fast_ma_values = [10, 15, 20, 25]
slow_ma_values = [40, 50, 60]

optimization_results = []

print("Running parameter optimization...")
for fast_ma in fast_ma_values:
    for slow_ma in slow_ma_values:
        if fast_ma >= slow_ma:
            continue
            
        # Update config
        test_config = config.copy()
        test_config['strategy']['fast_ma'] = fast_ma
        test_config['strategy']['slow_ma'] = slow_ma
        
        # Run backtest
        test_strategy = GoldMomentumStrategy(test_config)
        test_backtest = BacktestEngine(test_strategy, test_config)
        test_results = test_backtest.run(data)
        
        # Store results
        optimization_results.append({
            'fast_ma': fast_ma,
            'slow_ma': slow_ma,
            'total_return': test_results['metrics'].get('total_return_pct', 0),
            'sharpe_ratio': test_results['metrics'].get('sharpe_ratio', 0),
            'max_drawdown': test_results['metrics'].get('max_drawdown_pct', 0),
            'num_trades': test_results['metrics'].get('total_trades', 0)
        })
        
        print(f"MA({fast_ma}/{slow_ma}): Return={test_results['metrics'].get('total_return_pct', 0):.2f}%")

# Create results dataframe
opt_df = pd.DataFrame(optimization_results)
opt_df = opt_df.sort_values('total_return', ascending=False)

print("\nTop 5 Parameter Combinations:")
print(opt_df.head())

In [ ]:
# Visualize optimization results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Return vs Sharpe Ratio
scatter = axes[0].scatter(opt_df['sharpe_ratio'], opt_df['total_return'], 
                         c=opt_df['max_drawdown'], s=100, alpha=0.6, cmap='RdYlGn_r')
axes[0].set_title('Return vs Sharpe Ratio', fontsize=14)
axes[0].set_xlabel('Sharpe Ratio')
axes[0].set_ylabel('Total Return (%)')
axes[0].grid(True, alpha=0.3)
plt.colorbar(scatter, ax=axes[0], label='Max Drawdown (%)')

# Parameter combinations
opt_df['params'] = opt_df['fast_ma'].astype(str) + '/' + opt_df['slow_ma'].astype(str)
opt_df.nlargest(10, 'total_return').plot(x='params', y='total_return', 
                                          kind='bar', ax=axes[1], alpha=0.7)
axes[1].set_title('Top 10 Parameter Combinations by Return', fontsize=14)
axes[1].set_xlabel('MA Parameters (Fast/Slow)')
axes[1].set_ylabel('Total Return (%)')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Conclusion

This notebook demonstrated:
- Loading and analyzing Gold/USD historical data
- Calculating technical indicators
- Running backtests with the momentum strategy
- Analyzing performance metrics
- Optimizing strategy parameters

Next steps:
- Test the strategy in paper trading mode
- Monitor performance in real-time
- Consider additional indicators or strategies
- Implement machine learning enhancements